In [1]:

using Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))

using GeneralizedPerturbedEquilibrium
const FT = GeneralizedPerturbedEquilibrium.ForcingTerms
const read_coil_dat = FT.read_coil_dat
const compute_biot_savart_boundary! = FT.compute_biot_savart_boundary!

function export_hall_csv_notebook(;
    coil_file::String,
    out_csv::String = "hall_output.csv",
    current_A::Float64 = 100.0,
    R_inner_m::Float64 = 0.10,
    R_outer_m::Float64 = 0.15,
    nphi_inner::Int = 24,
    nz_inner::Int = 22,
    nphi_outer::Int = 20,
    nz_outer::Int = 10,
    zspan_inner::Float64 = 0.60,
    zspan_outer::Float64 = 0.40,
    z0::Float64 = 0.0
)

    cs = read_coil_dat(coil_file)
    cs.currents .= current_A
    coils = [cs]

    # --- Hall grid (cylindrical) ---
    φi = collect(range(0, 2π, length=nphi_inner+1)[1:end-1])
    φo = collect(range(0, 2π, length=nphi_outer+1)[1:end-1])

    Zi = collect(range(z0 - zspan_inner, z0 + zspan_inner; length=nz_inner))
    Zo = collect(range(z0 - zspan_outer, z0 + zspan_outer; length=nz_outer))

    N = nphi_inner*nz_inner + nphi_outer*nz_outer

    R = zeros(N)
    φ = zeros(N)
    Z = zeros(N)
    shell = zeros(Int, N)

    k = 1
    for iz in eachindex(Zi), ip in eachindex(φi)
        R[k] = R_inner_m
        φ[k] = φi[ip]
        Z[k] = Zi[iz]
        shell[k] = 1
        k += 1
    end

    for iz in eachindex(Zo), ip in eachindex(φo)
        R[k] = R_outer_m
        φ[k] = φo[ip]
        Z[k] = Zo[iz]
        shell[k] = 2
        k += 1
    end

    # --- Field evaluation ---
    BR = zeros(N)
    BP = zeros(N)
    BZ = zeros(N)

    compute_biot_savart_boundary!(BR, BP, BZ, R, φ, Z, coils)
    Bmag = sqrt.(BR.^2 .+ BP.^2 .+ BZ.^2)

    # --- Export CSV ---
    open(out_csv, "w") do io
        println(io, "x,y,z,R,phi,shell,B_R,B_phi,B_Z,B_mag")
        for i in 1:N
            x = R[i] * cos(φ[i])
            y = R[i] * sin(φ[i])
            println(io, "$(x),$(y),$(Z[i]),$(R[i]),$(φ[i]),$(shell[i]),$(BR[i]),$(BP[i]),$(BZ[i]),$(Bmag[i])")
        end
    end

    out_csv
end

  Activating project at `~/Documents/GitHub/JULIA_GPEC`


export_hall_csv_notebook (generic function with 1 method)

In [5]:
include("tilt_shift.jl")

phys_file  = "sparc_pf1u.dat"
field_file = "out_3mm.dat"

# ---------------------------------------------------------
# 1) generate shifted coil
# ---------------------------------------------------------
transform_coil(
    phys_file,
    field_file;
    shift_x    = 0.000,
    shift_y    = 0.000,
    shift_z    = 0.000,
    tilt_x_deg = 0.1,
    tilt_y_deg = 0.0,
    tilt_z_deg = 0.0,
)

# ---------------------------------------------------------
# 2) Hall data from shifted coil
# ---------------------------------------------------------
export_hall_csv_notebook(
    coil_file = field_file,
    out_csv   = "hall_probe_example.csv",
    current_A = 100.0
)

# ---------------------------------------------------------
# 3) full pipeline (physical + field)
# ---------------------------------------------------------
include("coil_centering_full.jl")

results = run_coil_axis_comparison(
    run_name             = "out_3mm",
    script_dir           = "./",
    coil_current_a       = 100.0,
    physical_coil_file   = phys_file,

    field_hall_mode      = :internal,
    field_hall_input_csv = "hall_probe_example.csv",

    physical_hall_mode   = :internal,
    display_axis_plot    = false
)

# ---------------------------------------------------------
# 4) extract comparable objects
# ---------------------------------------------------------
phys = results.physical_corrected
fld  = results.field_corrected

# ---------------------------------------------------------
# 5) one-line difference (FIELD - PHYSICAL)
# ---------------------------------------------------------
dx  = (fld.x0 - phys.x0) * 1e3
dy  = (fld.y0 - phys.y0) * 1e3
dz  = (fld.z0 - phys.z0) * 1e3
dtx = fld.tilt_x - phys.tilt_x
dty = fld.tilt_y - phys.tilt_y

@printf(
    "ΔFIELD-PHYSICAL: dx=%+.4f mm  dy=%+.4f mm  dz=%+.4f mm  dtx=%+.6f°  dty=%+.6f°\n",
    dx, dy, dz, dtx, dty
)


Loaded PHYSICAL coil:


  Activating project at `~/Documents/GitHub/JULIA_GPEC`


  file = sparc_pf1u.dat
  ncoil=1, s=1, nsec=30600, I=100.0 A

Loaded FIELD coil:
  file = ./out_3mm.dat
  ncoil=1, s=1, nsec=30600, I=100.0 A

Physical geom:
  center = (-2.423, 1.082, 2307.411) mm
  tilt_x = 0.047918 deg, tilt_y = 0.102029 deg
  fit radius = 928.829 mm
  R range = [705.437, 1256.983] mm
  Z range = [2185.500, 3033.600] mm

Field geom:
  center = (-2.423, 1.082, 2307.411) mm
  tilt_x = 0.147910 deg, tilt_y = 0.102008 deg
  fit radius = 928.829 mm
  R range = [705.240, 1256.832] mm
  Z range = [2183.700, 3035.200] mm

Computing synthetic Hall data from PHYSICAL coil:
  probes = 728
  R1 = 282.175 mm, Z1=[1707.411, 2907.411] mm
  R2 = 628.491 mm, Z2=[1907.411, 2707.411] mm

Exported Hall CSV: ./hall_probe_out_3mm_PHYSICAL.csv

B_R zero-crossing sinusoid, PHYSICAL, shell 1:
  valid = 24/24, Z0 = 2307.128520 mm, tilt_x = -0.011056 deg, tilt_y = -0.033941 deg

B_R zero-crossing sinusoid, PHYSICAL, shell 2:
  valid = 20/20, Z0 = 2307.664398 mm, tilt_x = 0.058041 deg, tilt_y

In [32]:
using Printf
# ---------------------------------------------------------
# 0) Setup suppression
# ---------------------------------------------------------
# Open a null device to swallow the output
out = devnull 

include("tilt_shift.jl")

phys_file  = "sparc_pf1u.dat"
field_file = "out_3mm.dat"

# Wrap all noisy functions in this block
redirect_stdout(out) do
    # ---------------------------------------------------------
    # 1) generate shifted coil
    # ---------------------------------------------------------
    transform_coil(
        phys_file,
        field_file;
        shift_x    = 0.000,
        shift_y    = 0.000,
        shift_z    = 0.000,
        tilt_x_deg = 0.1,
        tilt_y_deg = 0.0,
        tilt_z_deg = 0.0,
    )

    # ---------------------------------------------------------
    # 2) Hall data from shifted coil
    # ---------------------------------------------------------
    export_hall_csv_notebook(
        coil_file = field_file,
        out_csv   = "hall_probe_example.csv",
        current_A = 100.0
    )

    # ---------------------------------------------------------
    # 3) full pipeline (physical + field)
    # ---------------------------------------------------------
    include("coil_centering_full.jl")

    global results = run_coil_axis_comparison(
        run_name             = "out_3mm",
        script_dir           = "./",
        coil_current_a       = 100.0,
        physical_coil_file   = phys_file,
        field_hall_mode      = :internal,
        field_hall_input_csv = "hall_probe_example.csv",
        physical_hall_mode   = :internal,
        display_axis_plot    = false
    )
end # End of suppression

# ---------------------------------------------------------
# 4) extract comparable objects
# ---------------------------------------------------------
phys = results.physical_corrected
fld  = results.field_corrected

# ---------------------------------------------------------
# 5) one-line difference (FIELD - PHYSICAL)
# ---------------------------------------------------------
dx  = (fld.x0 - phys.x0) * 1e3
dy  = (fld.y0 - phys.y0) * 1e3
dz  = (fld.z0 - phys.z0) * 1e3
dtx = fld.tilt_x - phys.tilt_x
dty = fld.tilt_y - phys.tilt_y

# This will print normally because it is outside the redirect_stdout block
@printf(
    "ΔFIELD-PHYSICAL: dx=%+.4f mm  dy=%+.4f mm  dz=%+.4f mm  dtx=%+.6f°  dty=%+.6f°\n",
    dx, dy, dz, dtx, dty
)

  Activating project at `~/Documents/GitHub/JULIA_GPEC`


ΔFIELD-PHYSICAL: dx=-0.0001 mm  dy=-0.0000 mm  dz=+0.0004 mm  dtx=+0.099995°  dty=-0.000024°


In [33]:
using Printf

function run_scan(; shift_x=0.003, shift_y=0.005, shift_z=0.007, tilt_x_deg=0.0, tilt_y_deg=0.0, current_A = 100.0)
    # ---------------------------------------------------------
    # 0) Setup suppression
    # ---------------------------------------------------------
    # Open a null device to swallow the output
    out = devnull 

    include("tilt_shift.jl")

    phys_file  = "sparc_pf1u.dat"
    field_file = "coil_out.dat"

    # Wrap all noisy functions in this block
    redirect_stdout(out) do
        # ---------------------------------------------------------
        # 1) generate shifted coil
        # ---------------------------------------------------------
        transform_coil(
            phys_file,
            field_file;
            shift_x    = shift_x,
            shift_y    = shift_y,
            shift_z    = shift_z,
            tilt_x_deg = tilt_x_deg,
            tilt_y_deg = tilt_y_deg,
            tilt_z_deg = 0.0,
        )

        # ---------------------------------------------------------
        # 2) Hall data from shifted coil
        # ---------------------------------------------------------
        export_hall_csv_notebook(
            coil_file = field_file,
            out_csv   = "hall_probe_out.csv",
            current_A = current_A
        )

        # ---------------------------------------------------------
        # 3) full pipeline (physical + field)
        # ---------------------------------------------------------
        include("coil_centering_full.jl")

        global results = run_coil_axis_comparison(
            run_name             = "coil_out",
            script_dir           = "./",
            coil_current_a       = current_A,
            physical_coil_file   = phys_file,
            field_hall_mode      = :internal,
            field_hall_input_csv = "hall_probe_out.csv",
            physical_hall_mode   = :internal,
            display_axis_plot    = false
        )
    end # End of suppression

    # ---------------------------------------------------------
    # 4) extract comparable objects
    # ---------------------------------------------------------
    phys = results.physical_corrected
    fld  = results.field_corrected

    # ---------------------------------------------------------
    # 5) one-line difference (FIELD - PHYSICAL)
    # ---------------------------------------------------------
    dx  = (fld.x0 - phys.x0) * 1e3
    dy  = (fld.y0 - phys.y0) * 1e3
    dz  = (fld.z0 - phys.z0) * 1e3
    dtx = fld.tilt_x - phys.tilt_x
    dty = fld.tilt_y - phys.tilt_y

    # This will print normally because it is outside the redirect_stdout block
    @printf(
        "ΔFIELD-PHYSICAL: dx=%+.4f mm  dy=%+.4f mm  dz=%+.4f mm  dtx=%+.6f°  dty=%+.6f°\n",
        dx, dy, dz, dtx, dty
    )
end

run_scan (generic function with 7 methods)

In [36]:
# run scan with x shifts of 1, 3, 5, 10 mnm:
for shift in [0.01, 0.03, 0.05, 0.10, 1, 10, 50]
    run_scan(shift_x=.001, shift_y=0.005, shift_z=0.007, tilt_x_deg=shift, tilt_y_deg=0.0, current_A = 100.0)
end

  Activating project at `~/Documents/GitHub/JULIA_GPEC`


ΔFIELD-PHYSICAL: dx=+1.0000 mm  dy=+4.9998 mm  dz=+7.0000 mm  dtx=+0.009623°  dty=+0.000001°


  Activating project at `~/Documents/GitHub/JULIA_GPEC`


ΔFIELD-PHYSICAL: dx=+1.0000 mm  dy=+5.0002 mm  dz=+7.0008 mm  dtx=+0.029849°  dty=+0.000020°


  Activating project at `~/Documents/GitHub/JULIA_GPEC`


...

ΔFIELD-PHYSICAL: dx=+0.9999 mm  dy=+5.0006 mm  dz=+7.0042 mm  dtx=+1.000018°  dty=+0.000017°


  Activating project at `~/Documents/GitHub/JULIA_GPEC`


ΔFIELD-PHYSICAL: dx=+1.0159 mm  dy=+4.8223 mm  dz=+7.1358 mm  dtx=+9.979791°  dty=+0.001646°


  Activating project at `~/Documents/GitHub/JULIA_GPEC`


ΔFIELD-PHYSICAL: dx=+1.0047 mm  dy=+4.6185 mm  dz=+7.3110 mm  dtx=+50.117132°  dty=+0.051223°


In [ ]:
include("coil_centering_full.jl")

results = run_coil_axis_comparison(
    run_name       = "sparc_pf1u_357_test",
    script_dir     = "./",
    coil_current_a = 100.0,
    field_hall_mode = :internal,
    display_axis_plot = false,   # suppress readline() in notebook
)
println("Results: ", results.field_corrected)


Physical/Field Coil Magnetic Axis Comparison
Physical baseline coil file:
  ./sparc_pf1u.dat
Field calculation coil file:
  ./sparc_pf1u_357_test.dat

Hall input modes:
  PHYSICAL_HALL_MODE = internal
  FIELD_HALL_MODE    = internal

Loaded PHYSICAL BASELINE coil:
  file = ./sparc_pf1u.dat
  ncoil=1, s=1, nsec=30600, I=100.0 A

Loaded FIELD CALCULATION coil:
  file = ./sparc_pf1u_357_test.dat
  ncoil=1, s=1, nsec=30600, I=100.0 A

Loaded BIAS-CORRECTION MODEL coil:
  file = ./sparc_pf1u.dat
  ncoil=1, s=1, nsec=30600, I=100.0 A

Physical baseline geometric axis:
  center = (-2.423, 1.082, 2307.411) mm
  tilt_x = 0.047918 deg, tilt_y = 0.102029 deg
  fit radius = 928.829 mm
  R range = [705.437, 1256.983] mm
  Z range = [2185.500, 3033.600] mm

Field-coil geometric axis:
  center = (0.712, 6.240, 2314.240) mm
  tilt_x = 0.340554 deg, tilt_y = 0.606225 deg
  fit radius = 928.829 mm
  R range = [709.239, 1258.091] mm
  Z range = [2181.700, 3052.600] mm

Computing synthetic Hall data from

  Activating project at `~/Documents/GitHub/JULIA_GPEC`


In [ ]:
using Printf

# Include once at the top for performance
include("tilt_shift.jl")
include("coil_centering_full.jl")

function run_scan(; 
    shift_x=0.0, shift_y=0.0, shift_z=0.0, 
    tilt_x_deg=0.0, tilt_y_deg=0.0, 
    current_A = 100.0,
    # Grid parameters added to the signature
    n_phi_inner = 24,
    n_z_inner   = 22
)
    phys_file  = "sparc_pf1u.dat"
    field_file = "coil_out.dat"
    hall_csv   = "A_Hall_Test.csv"

    local_results = nothing

    redirect_stdout(devnull) do
        # 1) Generate coil geometry
        transform_coil(
            phys_file, field_file;
            shift_x=shift_x, shift_y=shift_y, shift_z=shift_z,
            tilt_x_deg=tilt_x_deg, tilt_y_deg=tilt_y_deg, tilt_z_deg=0.0,
        )

        # 2) Export Hall Data (must match the grid we are testing)
        export_hall_csv_notebook(
            coil_file = field_file,
            out_csv   = hall_csv,
            current_A = current_A
            # Note: ensure export_hall_csv_notebook uses the same N values if applicable
        )

        # 3) Full pipeline with dynamic grid resolution
        local_results = run_coil_axis_comparison(
            run_name                  = "res_scan",
            script_dir                = @__DIR__,
            coil_current_a            = current_A,
            physical_coil_file        = joinpath(@__DIR__,phys_file),
            field_coil_file           = joinpath(@__DIR__,field_file),

            # --- DYNAMIC GRID SCAN PARAMETERS ---
            hall_n_phi_inner          = n_phi_inner,
            hall_n_z_inner            = n_z_inner,
            hall_n_phi_outer          = Int(round(n_phi_inner * 0.8)), # scaling outer grid
            hall_n_z_outer            = Int(round(n_z_inner * 0.5)),
            
            hall_r_inner_frac         = 0.40,
            hall_r_outer_frac         = 0.50,
            hall_z_halfspan_inner_m   = 0.60,
            hall_z_halfspan_outer_m   = 0.40,

            field_hall_mode           = :internal,
            field_hall_input_csv      = joinpath(@__DIR__,hall_csv),
            physical_hall_mode        = :internal,

            # Fitting controls
            tilt_calibration_radius_m = NaN,
            br_zero_noise_frac        = .0,#7,
            br_min_frac_for_fourier_xy= 0.10,
            bphi_sign                 = 1.0,
            xy_min_step0_m            = 0.050,
            xy_min_tol_m              = 1e-8,
            bias_correction_n_iter    = 4,
            bias_correction_damping   = 0.6,

            enable_axis_plot          = false,
            display_axis_plot         = false
        )
    end 

    phys = local_results.physical_corrected
    fld  = local_results.field_corrected

    dx = (fld.x0 - phys.x0) * 1e3
    dy = (fld.y0 - phys.y0) * 1e3
    
    # Calculate Total Position Error (Euclidean distance in XY)
    dr = sqrt(dx^2 + dy^2)

    @printf(
        "Grid: [%2d x %2d] | Probes: %3d | dx=%+.4f mm, dy=%+.4f mm | Error DR: %.4f mm\n",
        n_phi_inner, n_z_inner, (n_phi_inner*n_z_inner), dx, dy, dr
    )
    
    return dr
end



  Activating project at `~/Documents/GitHub/JULIA_GPEC`


run_scan (generic function with 7 methods)

In [66]:
# Test grid sizes from 8x8 up to 48x48
resolutions = [24]#, 16, 32, 48, 64]
for N in resolutions
    run_scan(
        shift_x=0.03, 
        shift_y=0.05,
        shift_z=0.07, 
        tilt_x_deg=0.0, 
        n_phi_inner=N, 
        n_z_inner=N
    )
end

┌ Warning: bias_correct_axis: measured values contain NaN — skipping correction
└ @ Main /Users/bursche/Documents/GitHub/JULIA_GPEC/examples/Br_3D_example/coil_centering_full.jl:548


Grid: [24 x 24] | Probes: 576 | dx=+32.1335 mm, dy=+47.4986 mm | Error DR: 57.3471 mm
